# Week 1 Day 4 Lab - Open-Source, Local, Token Cost

## Nine open-weight models on UnderCover's claims

We send 20 of the claims in the queue to nine open-weight models. Six of them run on OpenRouter, where several providers host each one, and three run on your own computer with Ollama. An optional tenth goes direct to Cerebras. Every model gets a smaller prompt than on Day 2. Then we put accuracy, cost and speed side by side.

## 4 parts:

PART 1: The data and a smaller prompt

PART 2: Six open-weight models on OpenRouter

PART 3: Three models on your computer with Ollama

PART 4: Accuracy, cost and speed side by side

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls paid models through OpenRouter and costs about 25 cents. If you're OK to spend that, you'll need some credit on your account at <a href="https://openrouter.ai">openrouter.ai</a>. If you'd rather not spend anything, there's a free version of this lab in <a href="week1day4free.ipynb">week1day4free.ipynb</a>. If your computer has no GPU, there's a CPU version in <a href="week1day4cpu.ipynb">week1day4cpu.ipynb</a>. It's free too, with local models small enough to run on the CPU.<br/><br/>
            Part 3 needs Ollama and about 8 GB of disk for the two small models, with 10 GB more for the optional third. The whole lab takes about 25 minutes to run. For a quicker run, set <code>TINY = True</code> in Part 1 to use 4 claims in place of 20.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Add print statements and try small adjustments to make the code your own.<br/><br/>
            Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Resources</h2>
            <span style="color:#118AB2;">The <a href="../guides/01_intro.ipynb">guides</a> folder in this repo has more information on topics like the command line, Git, Python and debugging.<br/><br/>
            If this lab is too much, consider courses 1, 2, 4 and 5 in <a href="https://edwarddonner.com/curriculum">my curriculum</a>: AI Builder, AI Coder, and the Core and Agentic Tracks of AI Engineer.<br/><br/>
            If it's too slow, put me on 2x and zip through it!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/connect.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Stay in touch</h2>
            <span style="color:#118AB2;">Please message me in Udemy anytime with questions, or email me at <a href="mailto:ed@edwarddonner.com">ed@edwarddonner.com</a>.<br/><br/>
            Let's connect on <a href="https://www.linkedin.com/in/eddonner/">LinkedIn</a>! And subscribe to my <a href="https://www.youtube.com/@edward.donner">YouTube channel</a> for more on LLMs, agents and coding agents.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import ollama
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import ContentFilterFinishReasonError, OpenAI, RateLimitError
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

## Part 1: The data and a smaller prompt

The seed data is in the repo's `data` folder, as on Day 2. Today we leave out the 5 manual claims. They depend on the claims manual or the customer's history, which the agent gets in Week 2, so the policy wording alone can't settle them. That leaves 20 claims. Some models take several minutes on all 20, so set `TINY = True` for a quicker run on 4 of them.

In [ ]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
claims = [claim for claim in claims if labels[claim["claim_id"]]["difficulty"] != "manual"]
policies = {policy["policy_id"]: policy for policy in load("policies.json")}

print(f"{len(claims)} claims")

In [ ]:
decisions = pd.DataFrame([labels[claim["claim_id"]] for claim in claims])
decisions["decision"].value_counts()

12 of the 20 are approvals, so a model that approves every claim would get 60% of the decisions right. Keep that number in mind when we look at the results.

### A smaller prompt

On Day 2 the prompt had the whole policy wording, and the policy record and the claim as indented JSON. Today we trim it in three ways:

1. A short version of the policy wording, in `data/policy_wording_short.md`
2. Only the sections of it that matter for the type of claim
3. The policy record and the claim as compact JSON, without the ids and without the text of the documents

A smaller prompt costs less, runs faster and suits a small model on your own computer. It can also leave out something the model needs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: tokens</h2>
            <span style="color:#118AB2;">Models read and write tokens, which are pieces of words. In English a token is about three quarters of a word.<br/><br/>
            Providers charge per million tokens, with one price for the tokens you send and another for the tokens the model writes. The tokens it writes usually cost several times more, and reasoning tokens count as written.
            </span>
        </td>
    </tr>
</table>

In [ ]:
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
short_wording = (DATA / "policy_wording_short.md").read_text(encoding="utf-8")

print(f"The policy wording is {len(policy_wording.split()):,} words")
print(f"The short version is {len(short_wording.split()):,} words")

The short wording has a section for each benefit, with headings that start with `## `. We split it into sections so we can pick the ones each claim needs.

In [ ]:
def split_sections(wording):
    """Split the wording into its opening lines and a dict of its sections, keyed by heading."""
    parts = wording.split("\n## ")
    sections = {part.split("\n")[0].strip(): "## " + part for part in parts[1:]}
    return parts[0], sections


opening, sections = split_sections(short_wording)
list(sections)

In [ ]:
BENEFITS = {
    "trip_cancellation": "Cancellation and Interruption",
    "trip_interruption": "Cancellation and Interruption",
    "trip_delay": "Trip Delay",
    "baggage_delay": "Baggage Delay",
    "baggage": "Baggage",
    "passport": "Passport",
    "emergency_medical": "Emergency Medical",
}


def make_wording(claim):
    """The sections of the short wording that this claim needs: the limits, its benefit, the add-ons if the policy has any, and the rules for every claim."""
    headings = ["Limits", BENEFITS[claim["claim_type"]]]
    if policies[claim["policy_id"]]["add_ons"]:
        headings.append("Add-ons")
    headings += ["Never covered", "Payment", "Claims"]
    return "\n\n".join([opening.strip()] + [sections[heading].strip() for heading in headings])

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""


def make_user_prompt(claim):
    """The policy record and the claim as compact JSON, without the ids and the text of the documents."""
    policy = policies[claim["policy_id"]]
    record = {key: value for key, value in policy.items() if key not in ["policy_id", "customer_id", "sold_through", "status"]}
    details = {key: value for key, value in claim.items() if key not in ["policy_id", "customer_id", "status"]}
    details["documents"] = [{"name": document["name"], "type": document["type"]} for document in claim["documents"]]
    return f"""# Policy record

{json.dumps(record)}

# Claim

{json.dumps(details)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording it needs, then the policy record and the claim."""
    return [
        {"role": "system", "content": f"{INSTRUCTIONS}\n{make_wording(claim)}\n"},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

Here's the whole prompt for the first claim, exactly as the model sees it.

In [ ]:
messages = make_messages(claims[0])
print(messages[0]["content"])
print(messages[1]["content"])

## Part 2: Six open-weight models on OpenRouter

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Open-weight models on OpenRouter</h2>
            <span style="color:#118AB2;">An open-weight model's weights are published, so anyone with the hardware can serve it. On OpenRouter most open-weight models have several providers, at different prices, speeds and precisions.<br/><br/>
            With <code>require_parameters</code> OpenRouter only picks providers that support every parameter we send, including Structured Outputs. The response says which provider answered.
            </span>
        </td>
    </tr>
</table>

We use the same Pydantic class as on Day 2 for Structured Outputs. The rationale comes before the decision so the model works through the claim before it decides.

In [ ]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

### Calling the LLM

`decide` is Day 2's function with two more things in the result: the provider that answered and the number of input tokens.

Some providers run a content filter over the replies, and now and then one blocks an answer. `parse` then raises `ContentFilterFinishReasonError`. We count that claim as wrong and carry on with the rest.

In [ ]:
def decide(claim, model, effort, ignore=None):
    """Ask one model on OpenRouter to decide one claim, skipping any providers listed in ignore."""
    start = time.time()
    provider = {"require_parameters": True}
    if ignore:
        provider["ignore"] = ignore
    try:
        response = openrouter.chat.completions.parse(
            model=model,
            messages=make_messages(claim),
            response_format=ClaimDecision,
            reasoning_effort=effort,
            extra_body={"provider": provider},
        )
        answer = response.choices[0].message.parsed
        decision, payout, rationale = answer.decision, answer.payout, answer.rationale
    except ContentFilterFinishReasonError as error:
        print(f"{model} on {claim['claim_id']}: the provider's content filter blocked the reply")
        response = error.completion
        decision, payout, rationale = "blocked", 0.0, "The provider's content filter blocked the reply"
    return {
        "model": model,
        "effort": effort,
        "where": "OpenRouter",
        "claim_id": claim["claim_id"],
        "decision": decision,
        "payout": payout,
        "rationale": rationale,
        "provider": response.provider,
        "input_tokens": response.usage.prompt_tokens,
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
first = decide(claims[0], "deepseek/deepseek-v4.1-flash", "low")
first

### What the smaller prompt saves

Here's the same claim with Day 2's prompt: the whole policy wording, and the policy record and the claim as indented JSON.

In [ ]:
def make_day2_messages(claim):
    """Day 2's prompt: the whole policy wording, and the policy record and the claim as indented JSON."""
    policy = policies[claim["policy_id"]]
    return [
        {"role": "system", "content": f"{INSTRUCTIONS}\n{policy_wording}\n"},
        {"role": "user", "content": f"# Policy record\n\n{json.dumps(policy, indent=2)}\n\n# Claim\n\n{json.dumps(claim, indent=2)}\n"},
    ]


day2 = openrouter.chat.completions.parse(
    model="deepseek/deepseek-v4.1-flash",
    messages=make_day2_messages(claims[0]),
    response_format=ClaimDecision,
    reasoning_effort="low",
    extra_body={"provider": {"require_parameters": True}},
)

print(f"Day 2's prompt: {day2.usage.prompt_tokens:,} input tokens")
print(f"Today's prompt: {first['input_tokens']:,} input tokens")

### Now running the six models

This time we record whether each model gets the decision right, and whether it gets the decision and the payout right.

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def score(answer, label):
    """Add the difficulty, and whether the decision, and the decision with the payout, match the handler's."""
    return {
        **answer,
        "difficulty": label["difficulty"],
        "decision_right": answer["decision"] == label["decision"],
        "right": is_right(answer, label),
    }


def report(results):
    """Print how a run did: the decisions right, the decisions right with the payout too, the cost and the time."""
    decisions_right = sum(result["decision_right"] for result in results)
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{results[0]['model']}: {decisions_right} of {len(results)} decisions right, {right} with the payout too, ${cost:.4f} in {seconds:.0f} seconds")

In [ ]:
def run(model, effort, claims, labels, ignore=None):
    """Run a model on OpenRouter on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide(claim, model, effort, ignore)
        results.append(score(answer, labels[claim["claim_id"]]))
    report(results)
    return results

Each model runs at the lowest reasoning level it offers:

| Model | From | Reasoning |
|---|---|---|
| MiMo V2.6 Pro | Xiaomi | Not set: it has no reasoning setting |
| GLM 5.3 Flash | Z.ai | `low`: it always reasons, and low is its lowest level |
| Kimi K3 | Moonshot AI | `low` |
| DeepSeek V4.1 Flash | DeepSeek | `low` |
| Nemotron 3 Ultra | Nvidia | `medium`: its lowest level |
| Mistral Medium 3.5 | Mistral | `none`: it offers only high or none |

In [ ]:
mimo = run("xiaomi/mimo-v2.6-pro", None, claims, labels)

In our runs one of GLM 5.3 Flash's providers, Wafer, ignored the low reasoning setting and thought for up to 3,000 tokens on a single claim. We skip it by adding `ignore` to the provider settings.

In [ ]:
glm = run("z-ai/glm-5.3-flash", "low", claims, labels, ignore=["wafer"])

In [ ]:
kimi = run("moonshotai/kimi-k3", "low", claims, labels)

In [ ]:
deepseek = run("deepseek/deepseek-v4.1-flash", "low", claims, labels)

In [ ]:
nemotron = run("nvidia/nemotron-3-ultra-550b-a55b", "medium", claims, labels)

In [ ]:
mistral = run("mistralai/mistral-medium-3-5", "none", claims, labels)

In [ ]:
# Add your own experiments here!

## Part 3: Three models on your computer with Ollama

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Ollama</h2>
            <span style="color:#118AB2;">Ollama runs open-weight models on your own computer. Install it from <a href="https://ollama.com">ollama.com</a> and start it. It runs in the background and serves models on your computer. The <code>ollama</code> Python package talks to it.<br/><br/>
            Models are downloaded with a pull. Ollama loads a model into memory when you first use it and unloads it after a few minutes without use.
            </span>
        </td>
    </tr>
</table>

We run three models:

- `qwen3.5:4b` is Qwen 3.5 with 4 billion parameters, a 3.4 GB download that fits a computer with 8 GB of memory
- `gemma4:e2b-it-qat` is Gemma 4 E2B, the instruction-tuned version trained to run at 4 bits, a 4.3 GB download that also fits 8 GB
- `gemma4:e4b` is Gemma 4 E4B, a 9.6 GB download that needs a computer with 16 GB of memory. It's also the slowest of the three, so it's optional

The pulls take a few minutes the first time.

In [ ]:
print(ollama.pull("qwen3.5:4b").status)
print(ollama.pull("gemma4:e2b-it-qat").status)

In [ ]:
# Optional: a 9.6 GB download, and the slowest of the three
print(ollama.pull("gemma4:e4b").status)

Ollama takes our Pydantic schema as `format` and holds the model's reply to it, so even a small model answers in valid JSON. `think=False` turns reasoning off to keep the small models quick. `num_ctx` sets the context window, which Ollama otherwise sets to 4,096 tokens whatever the model supports.

In [ ]:
def decide_locally(claim, model):
    """Ask a model on this computer to decide one claim, through Ollama."""
    start = time.time()
    response = ollama.chat(
        model=model,
        messages=make_messages(claim),
        format=ClaimDecision.model_json_schema(),
        think=False,
        options={"num_ctx": 8192},
    )
    answer = ClaimDecision.model_validate_json(response.message.content)
    return {
        "model": model,
        "effort": "none",
        "where": "This computer",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "input_tokens": response.prompt_eval_count,
        "cost": 0.0,
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
def run_locally(model, claims, labels):
    """Run a model on this computer on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide_locally(claim, model)
        results.append(score(answer, labels[claim["claim_id"]]))
    report(results)
    return results

In [ ]:
first_local = decide_locally(claims[0], "qwen3.5:4b")
first_local

Now the 20 claims on each model. Each call runs on your own computer, so the time depends on your hardware.

In [ ]:
qwen = run_locally("qwen3.5:4b", claims, labels)

In [ ]:
gemma_e2b = run_locally("gemma4:e2b-it-qat", claims, labels)

In [ ]:
# Optional: only if you pulled gemma4:e4b
gemma_e4b = run_locally("gemma4:e4b", claims, labels)

## Optional: a tenth model direct on Cerebras

OpenRouter passes each request on to a provider. You can also skip OpenRouter and call a provider directly with an account of your own. Cerebras builds its own chips for inference and serves Qwen 3.8 27B at about 1,850 tokens a second. It's our tenth open-weight model.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">A Cerebras account</h2>
            <span style="color:#118AB2;">This part is optional and needs an API key from Cerebras. Sign up at <a href="https://cloud.cerebras.ai">cloud.cerebras.ai</a>, create a key under API Keys and add it to the <code>.env</code> file as <code>CEREBRAS_API_KEY</code>. Pay-as-you-go accounts start with $5 of free credit. The 20 claims cost about 4 cents.<br/><br/>
            If you skip this part, take <code>qwen_cerebras</code> out of the list at the start of Part 4.
            </span>
        </td>
    </tr>
</table>

In [ ]:
cerebras_api_key = os.getenv("CEREBRAS_API_KEY")
CEREBRAS_BASE_URL = "https://api.cerebras.ai/v1"

if cerebras_api_key:
    print("Cerebras API key found")
else:
    print("No Cerebras API key found. Add CEREBRAS_API_KEY to the .env file")

cerebras = OpenAI(base_url=CEREBRAS_BASE_URL, api_key=cerebras_api_key, timeout=120, max_retries=5)

We use the OpenAI client again, pointed at Cerebras, with the same prompt and the same Structured Outputs class.

Cerebras doesn't report the cost of a call. We work it out from the token counts and its prices on 28 September 2026: $0.99 per million input tokens and $1.49 per million output tokens.

We run it at `none`, its lowest reasoning level. It also offers `low`, `medium` and `high`.

Now and then Cerebras is too busy to take a request. It answers with a 429 error and tells the OpenAI client not to retry. `decide_on_cerebras` waits 5 seconds and tries again, up to 5 times. The clock for the claim starts again with each try.

In [ ]:
CEREBRAS_INPUT_PRICE = 0.99 / 1_000_000
CEREBRAS_OUTPUT_PRICE = 1.49 / 1_000_000


def decide_on_cerebras(claim, model, effort):
    """Ask a model on Cerebras to decide one claim, trying again when Cerebras is too busy."""
    for attempt in range(5):
        start = time.time()
        try:
            response = cerebras.chat.completions.parse(
                model=model,
                messages=make_messages(claim),
                response_format=ClaimDecision,
                reasoning_effort=effort,
            )
            break
        except RateLimitError:
            if attempt == 4:
                raise
            print(f"Cerebras is busy. Trying {claim['claim_id']} again in 5 seconds")
            time.sleep(5)
    answer = response.choices[0].message.parsed
    usage = response.usage
    return {
        "model": model,
        "effort": effort,
        "where": "Cerebras",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "input_tokens": usage.prompt_tokens,
        "cost": usage.prompt_tokens * CEREBRAS_INPUT_PRICE + usage.completion_tokens * CEREBRAS_OUTPUT_PRICE,
        "seconds": round(time.time() - start, 1),
    }


def run_on_cerebras(model, effort, claims, labels):
    """Run a model on Cerebras on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide_on_cerebras(claim, model, effort)
        results.append(score(answer, labels[claim["claim_id"]]))
    report(results)
    return results

In [ ]:
qwen_cerebras = run_on_cerebras("qwen-3.8-27b", "none", claims, labels)

When I ran this, it took about a second a claim.

## Part 4: Accuracy, cost and speed side by side

One row for each model. Accuracy is the percentage of the 20 claims it got right, first on the decision alone and then on the decision and the payout together.

In [ ]:
# If you skipped Gemma 4 E4B or Cerebras, take gemma_e4b or qwen_cerebras out of this list
experiments = [mimo, glm, kimi, deepseek, nemotron, mistral, qwen_cerebras, qwen, gemma_e2b, gemma_e4b]

rows = []
for results in experiments:
    count = len(results)
    rows.append({
        "model": results[0]["model"].split("/")[-1],
        "where": results[0]["where"],
        "reasoning": results[0]["effort"] or "not set",
        "decision right (%)": round(100 * sum(result["decision_right"] for result in results) / count),
        "decision and payout right (%)": round(100 * sum(result["right"] for result in results) / count),
        "seconds a claim": round(sum(result["seconds"] for result in results) / count, 1),
        "cost a claim ($)": round(sum(result["cost"] for result in results) / count, 5),
    })

summary = pd.DataFrame(rows)
summary

### Accuracy against cost

The best place to be is the top left: accurate and cheap. The local models cost nothing to run, so they sit at the left edge.

In [ ]:
fig = px.scatter(
    summary,
    x="cost a claim ($)",
    y="decision and payout right (%)",
    color="where",
    text="model",
    title="Accuracy against cost",
    height=600,
)
fig.update_traces(marker_size=14, textposition="top center")
fig.update_yaxes(range=[0, 100])
fig.show()

### Accuracy against speed

Again the top left is best: accurate and fast. The local times depend on your computer.

In [ ]:
fig = px.scatter(
    summary,
    x="seconds a claim",
    y="decision and payout right (%)",
    color="where",
    text="model",
    title="Accuracy against speed",
    height=600,
)
fig.update_traces(marker_size=14, textposition="top center")
fig.update_yaxes(range=[0, 100])
fig.show()

### All three at once

Drag the chart to turn it.

In [ ]:
fig = px.scatter_3d(
    summary,
    x="cost a claim ($)",
    y="seconds a claim",
    z="decision and payout right (%)",
    color="where",
    text="model",
    title="Accuracy, cost and speed",
    height=700,
)
fig.show()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Every hosted model here costs under a cent a claim, and the local models cost nothing beyond the computer and its power. So the choice between these models is mostly about accuracy and speed.<br/><br/>
            Customers will also ask where their data goes. With a local model it stays on hardware you control.
            </span>
        </td>
    </tr>
</table>

### The claims the models got wrong

This counts how many of the models got each claim right, fewest first.

In [ ]:
scored = pd.concat([pd.DataFrame(results) for results in experiments])
scored.groupby("claim_id")["right"].sum().sort_values().head(8)

When I ran this, none of the models got CLM-1016 right. Here's the claim and the handler's decision.

In [ ]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
claim = [claim for claim in queue if claim["claim_id"] == "CLM-1016"][0]
decision = queue_labels["CLM-1016"]

display(Markdown(f"""**{claim['summary']}**

{claim['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

`show_misses` shows each claim a model got wrong, with its rationale, the handler's reason and what the claim is there to test. Pass in any set of results.

In [ ]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))

In [ ]:
show_misses(deepseek, labels)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">None of the models got CLM-1016 right, a claim the handler declined. The reason is in the discharge summary, and our smaller prompt sends the names of the documents without their text.<br/><br/>
            Challenge: change <code>make_user_prompt</code> so that it sends the text of the documents, then run two or three of the models again. How many of them decline CLM-1016 now, and how many more input tokens does each claim use?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#069D75;">Congratulations!</h2>
            <span style="color:#069D75;">You've compared nine open-weight models on UnderCover's claims, six on OpenRouter and three on your own computer, with accuracy, cost and speed side by side.<br/><br/>
            On Day 5 we look at harness engineering.
            </span>
        </td>
    </tr>
</table>